# Adaptive GRAPE parameter fitting experiment

This notebook keeps the same readable style as `test_grape.ipynb`, but now simulates a closed-loop calibration step.

Goal for this first version:

1. Optimize a pulse with unitary GRAPE using the nominal physics model.
2. Fine-tune that pulse with non-unitary GRAPE using the nominal physics model.
3. Generate 500 nearby pulses and measure them on a hidden true model.
4. Fit Hamiltonian parameters and the measurement-response line `A*x + B` from those measured data.

Nothing is saved for hardware yet. This notebook is only to check whether the adaptive calibration pieces behave sensibly.


In [ ]:
from pathlib import Path
import json
import sys

import numpy as np
import jax
jax.config.update("jax_enable_x64", True)
import jax.numpy as jnp
import optax
import matplotlib.pyplot as plt
from tqdm.auto import tqdm

cwd = Path.cwd()
project_dir = cwd if (cwd / "grape.py").exists() else cwd / "Simplified_adaptive_grape"
repo_root = project_dir.parent
sys.path.insert(0, str(project_dir))
sys.path.insert(0, str(repo_root))

import toolbox as tbx
import grape


## Experimental values and the hidden true model

The nominal model reads the experimental values from `configuration.json`. The hidden true model adds small realistic mismatches:

- chi shifted by a few kHz,
- qubit frequency offset of `0.1 MHz`,
- cavity frequency offset of `0.2 MHz`,
- cavity self-Kerr set to `-0.0007 MHz`,
- qubit drive scale `1.001`, cavity drive scale `0.98`,
- slightly shorter lifetimes.

All frequencies in the Hamiltonian below are angular frequencies in `rad / us`. Since `1 MHz = 1 cycle / us`, the conversion is `2*pi*MHz`.


In [ ]:
config_path = repo_root / "configuration.json"
with config_path.open("r") as f:
    cfg = json.load(f)

# Nominal values read directly from the JSON.
chi_nominal = -2 * jnp.pi * cfg["chi_kHz"] * 1e-3
cavity_self_kerr_nominal = 2 * jnp.pi * cfg["self_Kerr_kHz"] * 1e-3

qubit_T1_nominal = float(cfg["qubit_T1_us"])
qubit_T2_nominal = float(cfg["qubit_T2_us"])
cavity_T1_nominal = float(cfg["storage_T1_us"])
cavity_T2_nominal = float(cfg["storage_T2_us"])

mu_qub = 20.0
mu_cav = 20.0

# Hidden true model mismatches. These are the numbers you can tweak first.
chi_true_shift_MHz = 0.004                 # 4 kHz chi error
qubit_freq_shift_MHz = 0.100
cavity_freq_shift_MHz = 0.200
cavity_self_kerr_true_MHz = -0.0007
qubit_amp_factor_true = 1.001
cavity_amp_factor_true = 0.980

chi_true = chi_nominal + 2 * jnp.pi * chi_true_shift_MHz
qubit_shift_true = 2 * jnp.pi * qubit_freq_shift_MHz
cavity_shift_true = 2 * jnp.pi * cavity_freq_shift_MHz
cavity_self_kerr_true = 2 * jnp.pi * cavity_self_kerr_true_MHz

qubit_T1_true = 0.88 * qubit_T1_nominal
qubit_T2_true = 0.85 * qubit_T2_nominal
cavity_T1_true = 0.92 * cavity_T1_nominal
cavity_T2_true = 0.90 * cavity_T2_nominal

nominal_model = {
    "chi": chi_nominal,
    "qubit_shift": 0.0,
    "cavity_shift": 0.0,
    "cavity_self_kerr": cavity_self_kerr_nominal,
    "qubit_amp_factor": 1.0,
    "cavity_amp_factor": 1.0,
    "qubit_T1": qubit_T1_nominal,
    "qubit_T2": qubit_T2_nominal,
    "cavity_T1": cavity_T1_nominal,
    "cavity_T2": cavity_T2_nominal,
}

true_model = {
    "chi": chi_true,
    "qubit_shift": qubit_shift_true,
    "cavity_shift": cavity_shift_true,
    "cavity_self_kerr": cavity_self_kerr_true,
    "qubit_amp_factor": qubit_amp_factor_true,
    "cavity_amp_factor": cavity_amp_factor_true,
    "qubit_T1": qubit_T1_true,
    "qubit_T2": qubit_T2_true,
    "cavity_T1": cavity_T1_true,
    "cavity_T2": cavity_T2_true,
}

# Long selective-pi measurement response. If x is the 500-shot success fraction,
# observed = A*x + B maps 0 -> 0.01 and 1 -> 0.93.
A_true = 0.92
B_true = 0.01
shots_per_pulse = 500

print("nominal chi [rad/us]:", float(chi_nominal))
print("true chi shift [MHz]:", chi_true_shift_MHz)
print("true qubit/cavity shifts [MHz]:", qubit_freq_shift_MHz, cavity_freq_shift_MHz)
print("true A, B:", A_true, B_true)


## B-spline basis and system operators

This is the same shifted quadratic basis style as the simple GRAPE notebook: 20 basis functions, at most 3 overlapping, and coefficients bounded to `[-2, 2]`.


In [ ]:
N_cav = 25
target_n = 2
param_clip = 2.0
n_channels = 4

spline_degree = 2
bspln_num = 20
skip_left = spline_degree
skip_right = spline_degree
n_total_bsplines = bspln_num + skip_left + skip_right

delta_ns = 64
m = n_total_bsplines - spline_degree
T_ns = m * delta_ns
T_us = T_ns / 1000.0
single_basis_length_ns = (spline_degree + 1) * delta_ns

Nt = 80
time_start = 0.0
time_end = T_us
time_edges = jnp.linspace(time_start, time_end, Nt + 1)
time_mids = 0.5 * (time_edges[1:] + time_edges[:-1])
time_intervals = time_edges[1:] - time_edges[:-1]

bspline_builder = tbx.setup_bspline_builder(
    time_start,
    time_end,
    n_total_bsplines,
    spline_degree,
    skip_left,
    skip_right,
)
bsplns_mids = jnp.asarray(bspline_builder(np.asarray(time_mids)))
bsplns_edges = jnp.asarray(bspline_builder(np.asarray(time_edges)))

print("duration [ns]:", T_ns)
print("single basis pulse length [ns]:", single_basis_length_ns)
print("B-splines:", bsplns_mids.shape)
print("endpoint max:", float(jnp.max(jnp.abs(bsplns_edges[:, [0, -1]]))))
print("max active B-splines:", int(jnp.max(jnp.sum(bsplns_mids > 1e-12, axis=0))))

plt.figure(figsize=(9, 3))
for b in np.asarray(bsplns_edges):
    plt.plot(np.asarray(time_edges) * 1e3, b, lw=1)
plt.xlabel("time [ns]")
plt.ylabel("basis value")
plt.title("20 shifted quadratic B-splines")
plt.grid(alpha=0.3)
plt.show()


In [ ]:
a = tbx.tensor(tbx.identity(2), tbx.destroy(N_cav))
adag = tbx.hconj(a)
n_phot = adag @ a
n2_minus_n = n_phot @ n_phot - n_phot

sigz = tbx.tensor(tbx.sigma.z, tbx.identity(N_cav))
sigp = tbx.tensor(tbx.sigma.p, tbx.identity(N_cav))
sigm = tbx.hconj(sigp)
one = tbx.identity(2 * N_cav)
qubit_excited = 0.5 * (one - sigz)

psi_init = tbx.tensor(tbx.basis(2, 0), tbx.basis(N_cav, 0))
rho_init = psi_init @ tbx.hconj(psi_init)

print("Hilbert dimension:", psi_init.shape[0])


## Simulation functions

There are only two model probabilities:

- `unitary_probability`: Schrodinger evolution.
- `decay_probability`: the same Hamiltonian with T1/T2 collapse operators.

The cavity cable convention remains explicit: `e_cav -> i * conj(e_cav)`.


In [ ]:
def hamiltonian_tree(ctrl_coeffs, model):
    e_qub, e_cav = grape.controls_from_coefficients(ctrl_coeffs, bsplns_mids)
    e_qub = model["qubit_amp_factor"] * e_qub
    e_cav = model["cavity_amp_factor"] * 1j * jnp.conj(e_cav)

    H_drift = (
        model["chi"] * (n_phot @ qubit_excited)
        + 0.5 * model["cavity_self_kerr"] * n2_minus_n
        + model["cavity_shift"] * n_phot
        + model["qubit_shift"] * qubit_excited
    )

    return [
        [H_drift, 1.0, 1.0, 0.0],
        [sigp, mu_qub * e_qub, 1.0, 1.0],
        [adag, mu_cav * e_cav, 1.0, 1.0],
    ]


def collapse_ops(model):
    gamma_phi_qub = jnp.maximum(1.0 / model["qubit_T2"] - 0.5 / model["qubit_T1"], 0.0)
    gamma_phi_cav = jnp.maximum(1.0 / model["cavity_T2"] - 0.5 / model["cavity_T1"], 0.0)
    return [
        jnp.sqrt(1.0 / model["qubit_T1"]) * sigp,
        jnp.sqrt(2.0 * gamma_phi_qub) * qubit_excited,
        jnp.sqrt(1.0 / model["cavity_T1"]) * a,
        jnp.sqrt(2.0 * gamma_phi_cav) * n_phot,
    ]


def unitary_probability(ctrl_coeffs, model):
    psi_t = tbx.sesolve_htree(hamiltonian_tree(ctrl_coeffs, model), psi_init, time_intervals)
    return grape.fock_probability_from_state(psi_t[-1], N_cav, target_n)


def decay_probability(ctrl_coeffs, model):
    rho_final = tbx.mesolve_htree(
        hamiltonian_tree(ctrl_coeffs, model),
        collapse_ops(model),
        rho_init,
        time_intervals,
    )
    return grape.fock_probability_from_density(rho_final, N_cav, target_n)


## True experiment measurement

The hidden experiment first simulates the true physical probability `P_n`. Then it draws `500` binary shots with mean `P_n`. Finally it maps the measured fraction through `A*x+B`, with `A=0.92`, `B=0.01`.


In [ ]:
@jax.jit
def true_physical_probability(ctrl_coeffs):
    return decay_probability(ctrl_coeffs, true_model)


def measure_true_experiment(ctrl_coeffs, key):
    p_true = true_physical_probability(ctrl_coeffs)
    successes = jax.random.binomial(key, n=shots_per_pulse, p=jnp.clip(p_true, 0.0, 1.0))
    shot_fraction = successes / shots_per_pulse
    observed = A_true * shot_fraction + B_true
    return observed, shot_fraction, successes, p_true


## 1. Unitary GRAPE with the nominal physics model


In [ ]:
key = jax.random.key(10)
key, subkey = jax.random.split(key)
initial_coeffs = 0.03 * jax.random.normal(subkey, (n_channels, bspln_num))
initial_coeffs = grape.clip_coefficients(initial_coeffs, param_clip)

n_steps_unitary = 400
learning_rate_unitary = 0.03
optimizer_unitary = optax.adam(learning_rate_unitary)
opt_state_unitary = optimizer_unitary.init(initial_coeffs)

@jax.jit
def unitary_train_step(ctrl_coeffs, opt_state):
    def loss_fn(c):
        p = unitary_probability(c, nominal_model)
        return 1.0 - p + grape.pulse_penalty(c), p

    (loss_value, p_value), grads = jax.value_and_grad(loss_fn, has_aux=True)(ctrl_coeffs)
    updates, opt_state = optimizer_unitary.update(grads, opt_state)
    ctrl_coeffs = optax.apply_updates(ctrl_coeffs, updates)
    ctrl_coeffs = grape.clip_coefficients(ctrl_coeffs, param_clip)
    return ctrl_coeffs, opt_state, loss_value, p_value

ctrl_unitary = initial_coeffs
hist_unitary = []
pbar = tqdm(range(n_steps_unitary), desc="unitary nominal GRAPE")
for _ in pbar:
    ctrl_unitary, opt_state_unitary, loss_value, p_value = unitary_train_step(ctrl_unitary, opt_state_unitary)
    hist_unitary.append(float(p_value))
    pbar.set_postfix(P_n=f"{float(p_value):.6f}")

print("nominal unitary P_n:", float(unitary_probability(ctrl_unitary, nominal_model)))
print("nominal decay P_n:", float(decay_probability(ctrl_unitary, nominal_model)))
print("hidden true decay P_n:", float(true_physical_probability(ctrl_unitary)))


## 2. Fine-tune with non-unitary GRAPE

This still uses the nominal physics model, not the true model. The goal is just to account for the known nominal T1/T2 while optimizing the pulse.


In [ ]:
n_steps_decay = 250
learning_rate_decay = 0.015
optimizer_decay = optax.adam(learning_rate_decay)
opt_state_decay = optimizer_decay.init(ctrl_unitary)

@jax.jit
def decay_train_step(ctrl_coeffs, opt_state):
    def loss_fn(c):
        p = decay_probability(c, nominal_model)
        return 1.0 - p + grape.pulse_penalty(c), p

    (loss_value, p_value), grads = jax.value_and_grad(loss_fn, has_aux=True)(ctrl_coeffs)
    updates, opt_state = optimizer_decay.update(grads, opt_state)
    ctrl_coeffs = optax.apply_updates(ctrl_coeffs, updates)
    ctrl_coeffs = grape.clip_coefficients(ctrl_coeffs, param_clip)
    return ctrl_coeffs, opt_state, loss_value, p_value

ctrl_opt = ctrl_unitary
hist_decay = []
pbar = tqdm(range(n_steps_decay), desc="decay nominal GRAPE")
for _ in pbar:
    ctrl_opt, opt_state_decay, loss_value, p_value = decay_train_step(ctrl_opt, opt_state_decay)
    hist_decay.append(float(p_value))
    pbar.set_postfix(P_n=f"{float(p_value):.6f}")

print("nominal decay P_n after fine tune:", float(decay_probability(ctrl_opt, nominal_model)))
print("hidden true decay P_n after fine tune:", float(true_physical_probability(ctrl_opt)))


In [ ]:
plt.figure(figsize=(7, 3))
plt.plot(hist_unitary, label="unitary nominal GRAPE")
plt.plot(np.arange(len(hist_decay)) + len(hist_unitary), hist_decay, label="decay nominal fine tune")
plt.xlabel("optimizer step")
plt.ylabel(f"P_{target_n}")
plt.grid(alpha=0.3)
plt.legend()
plt.show()


## 3. Build a 500-point local experimental dataset

The dataset contains the optimized pulse plus nearby controls with small Gaussian coefficient perturbations. Each point is measured once with `500` shots on the hidden true model.


In [ ]:
dataset_size = 500
local_noise_std = 0.025

key, noise_key = jax.random.split(key)
noise = local_noise_std * jax.random.normal(noise_key, (dataset_size, n_channels, bspln_num))
controls_dataset = grape.clip_coefficients(ctrl_opt[None, :, :] + noise, param_clip)
controls_dataset = controls_dataset.at[0].set(ctrl_opt)

observed_dataset = []
shot_fraction_dataset = []
successes_dataset = []
true_probability_dataset = []

pbar = tqdm(range(dataset_size), desc="measure local pulses")
for i in pbar:
    key, measure_key = jax.random.split(key)
    observed, shot_fraction, successes, p_true = measure_true_experiment(controls_dataset[i], measure_key)
    observed_dataset.append(observed)
    shot_fraction_dataset.append(shot_fraction)
    successes_dataset.append(successes)
    true_probability_dataset.append(p_true)
    pbar.set_postfix(observed=f"{float(observed):.4f}", true=f"{float(p_true):.4f}")

observed_dataset = jnp.asarray(observed_dataset)
shot_fraction_dataset = jnp.asarray(shot_fraction_dataset)
successes_dataset = jnp.asarray(successes_dataset)
true_probability_dataset = jnp.asarray(true_probability_dataset)

print("controls_dataset:", controls_dataset.shape)
print("observed mean/std:", float(jnp.mean(observed_dataset)), float(jnp.std(observed_dataset)))
print("hidden true P_n mean/std:", float(jnp.mean(true_probability_dataset)), float(jnp.std(true_probability_dataset)))
print("best observed point:", float(jnp.max(observed_dataset)))
print("best hidden true point:", float(jnp.max(true_probability_dataset)))


In [ ]:
plt.figure(figsize=(6, 4))
plt.scatter(true_probability_dataset, observed_dataset, s=16, alpha=0.7)
plt.xlabel("hidden true physical P_n")
plt.ylabel("observed A*(successes/500)+B")
plt.title("local calibration dataset")
plt.grid(alpha=0.3)
plt.show()


## 4. Fit Hamiltonian parameters and the measurement response

We fit the nominal model to the measured dataset. The model prediction is

`observed_pred = A_fit * P_model(control; fitted Hamiltonian) + B_fit`.

The fitted raw vector is bounded with `tanh` so the optimizer cannot wander into absurd Hamiltonians during this first sanity check.


In [ ]:
def model_from_fit_raw(raw):
    chi_shift_MHz = 0.010 * jnp.tanh(raw[0])
    qubit_shift_MHz = 0.300 * jnp.tanh(raw[1])
    cavity_self_kerr_MHz = cfg["self_Kerr_kHz"] * 1e-3 + 0.002 * jnp.tanh(raw[2])
    cavity_shift_MHz = 0.400 * jnp.tanh(raw[3])

    qubit_amp_factor = 1.0 + 0.030 * jnp.tanh(raw[4])
    cavity_amp_factor = 1.0 + 0.080 * jnp.tanh(raw[5])

    qubit_T1_scale = jnp.exp(0.35 * jnp.tanh(raw[6]))
    qubit_T2_scale = jnp.exp(0.35 * jnp.tanh(raw[7]))
    cavity_T1_scale = jnp.exp(0.35 * jnp.tanh(raw[8]))
    cavity_T2_scale = jnp.exp(0.35 * jnp.tanh(raw[9]))

    A_fit = 0.90 + 0.15 * jnp.tanh(raw[10])
    B_fit = 0.02 + 0.04 * jnp.tanh(raw[11])

    model = {
        "chi": chi_nominal + 2 * jnp.pi * chi_shift_MHz,
        "qubit_shift": 2 * jnp.pi * qubit_shift_MHz,
        "cavity_shift": 2 * jnp.pi * cavity_shift_MHz,
        "cavity_self_kerr": 2 * jnp.pi * cavity_self_kerr_MHz,
        "qubit_amp_factor": qubit_amp_factor,
        "cavity_amp_factor": cavity_amp_factor,
        "qubit_T1": qubit_T1_nominal * qubit_T1_scale,
        "qubit_T2": qubit_T2_nominal * qubit_T2_scale,
        "cavity_T1": cavity_T1_nominal * cavity_T1_scale,
        "cavity_T2": cavity_T2_nominal * cavity_T2_scale,
    }
    response = {"A": A_fit, "B": B_fit}
    return model, response


def fitted_observed_prediction(raw, ctrl_coeffs):
    model, response = model_from_fit_raw(raw)
    p_model = decay_probability(ctrl_coeffs, model)
    return response["A"] * p_model + response["B"]


In [ ]:
fit_steps = 300
fit_batch_size = 24
fit_learning_rate = 0.030

fit_raw = jnp.zeros((12,))
fit_optimizer = optax.adam(fit_learning_rate)
fit_opt_state = fit_optimizer.init(fit_raw)

@jax.jit
def fit_step(fit_raw, fit_opt_state, batch_controls, batch_observed):
    def loss_fn(raw):
        pred = jax.vmap(lambda c: fitted_observed_prediction(raw, c))(batch_controls)
        return jnp.mean((pred - batch_observed) ** 2), pred

    (loss_value, pred), grads = jax.value_and_grad(loss_fn, has_aux=True)(fit_raw)
    updates, fit_opt_state = fit_optimizer.update(grads, fit_opt_state)
    fit_raw = optax.apply_updates(fit_raw, updates)
    return fit_raw, fit_opt_state, loss_value, jnp.mean(pred)

fit_loss_history = []
pbar = tqdm(range(fit_steps), desc="fit Hamiltonian + measurement response")
for step in pbar:
    key, batch_key = jax.random.split(key)
    batch_idx = jax.random.choice(batch_key, dataset_size, shape=(fit_batch_size,), replace=False)
    fit_raw, fit_opt_state, loss_value, mean_pred = fit_step(
        fit_raw,
        fit_opt_state,
        controls_dataset[batch_idx],
        observed_dataset[batch_idx],
    )
    fit_loss_history.append(float(loss_value))
    pbar.set_postfix(loss=f"{float(loss_value):.5e}", pred=f"{float(mean_pred):.4f}")


In [ ]:
def predict_dataset(raw):
    pieces = []
    for start in tqdm(range(0, dataset_size, fit_batch_size), desc="predict dataset"):
        stop = min(start + fit_batch_size, dataset_size)
        pieces.append(jax.vmap(lambda c: fitted_observed_prediction(raw, c))(controls_dataset[start:stop]))
    return jnp.concatenate(pieces)

pred_initial = predict_dataset(jnp.zeros((12,)))
pred_fitted = predict_dataset(fit_raw)

plt.figure(figsize=(12, 4))
plt.subplot(1, 2, 1)
plt.semilogy(fit_loss_history)
plt.xlabel("fit step")
plt.ylabel("MSE")
plt.title("parameter fit loss")
plt.grid(alpha=0.3)

plt.subplot(1, 2, 2)
plt.scatter(observed_dataset, pred_initial, s=14, alpha=0.5, label="initial model")
plt.scatter(observed_dataset, pred_fitted, s=14, alpha=0.5, label="fitted model")
lo = float(jnp.min(observed_dataset))
hi = float(jnp.max(observed_dataset))
plt.plot([lo, hi], [lo, hi], "k--", lw=1)
plt.xlabel("measured observed value")
plt.ylabel("model predicted observed value")
plt.legend()
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()

print("initial MAE:", float(jnp.mean(jnp.abs(pred_initial - observed_dataset))))
print("fitted MAE:", float(jnp.mean(jnp.abs(pred_fitted - observed_dataset))))


In [ ]:
fitted_model, fitted_response = model_from_fit_raw(fit_raw)

print("--- true parameters ---")
print("chi shift [MHz]:", chi_true_shift_MHz)
print("qubit shift [MHz]:", qubit_freq_shift_MHz)
print("cavity shift [MHz]:", cavity_freq_shift_MHz)
print("cavity self-Kerr [MHz]:", cavity_self_kerr_true_MHz)
print("qubit amp factor:", qubit_amp_factor_true)
print("cavity amp factor:", cavity_amp_factor_true)
print("qubit T1/T2 scale:", qubit_T1_true / qubit_T1_nominal, qubit_T2_true / qubit_T2_nominal)
print("cavity T1/T2 scale:", cavity_T1_true / cavity_T1_nominal, cavity_T2_true / cavity_T2_nominal)
print("A, B:", A_true, B_true)

print("\n--- fitted parameters ---")
print("chi shift [MHz]:", float((fitted_model["chi"] - chi_nominal) / (2 * jnp.pi)))
print("qubit shift [MHz]:", float(fitted_model["qubit_shift"] / (2 * jnp.pi)))
print("cavity shift [MHz]:", float(fitted_model["cavity_shift"] / (2 * jnp.pi)))
print("cavity self-Kerr [MHz]:", float(fitted_model["cavity_self_kerr"] / (2 * jnp.pi)))
print("qubit amp factor:", float(fitted_model["qubit_amp_factor"]))
print("cavity amp factor:", float(fitted_model["cavity_amp_factor"]))
print("qubit T1/T2 scale:", float(fitted_model["qubit_T1"] / qubit_T1_nominal), float(fitted_model["qubit_T2"] / qubit_T2_nominal))
print("cavity T1/T2 scale:", float(fitted_model["cavity_T1"] / cavity_T1_nominal), float(fitted_model["cavity_T2"] / cavity_T2_nominal))
print("A, B:", float(fitted_response["A"]), float(fitted_response["B"]))
